# 02 · Supervised fine-tuning (SFT)

Same loss as pre-training. Two changes:
1. **Data** is now (prompt, ideal response) pairs, formatted with a chat template.
2. **Weights**: 0 on the prompt, 1 on the response. We only learn to *answer*, not to *ask*.

Task for the rest of the talk: GSM8K math, answered in `\boxed{}` **within 256 tokens**.
The GSM8K reference solutions are short, so imitating them teaches brevity.

In [1]:
from common import *
from tinker_cookbook.hyperparam_utils import get_lr

train = load_gsm8k("train").shuffle(seed=0)
BATCH_SIZE, STEPS, LR = 32, 40, get_lr(MODEL)

ex = train[0]
show("question", ex["question"])
show("reference solution (reformatted)", gsm8k_solution(ex["answer"]))

/home/ray/default/post-training-webinar/.venv/lib/python3.12/site-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(



── question ──
Mustafa buys a fridge, a TV and a computer with a total budget of $1600. The TV costs $600 and the fridge costs $500 more than the computer. How much does the computer cost?


── reference solution (reformatted) ──
The fridge and computer cost 1600-600=1000 dollars.
The computer cost (1000-500)/2=250 dollars. \boxed{250}



In [5]:
# ── LIVE: the only real change from 01 is the weights ─────────────────────
def sft_datum(question, answer):
    convo = gsm8k_convo(question) + [{"role": "assistant", "content": gsm8k_solution(answer)}]
    model_input, weights = renderer.build_supervised_example(
        convo, train_on_what=renderers.TrainOnWhat.LAST_ASSISTANT_MESSAGE
    )
    return make_datum(model_input.to_ints(), weights.tolist())

In [6]:
# Look at what the mask does: green = trained on, grey = context only.
d = sft_datum(ex["question"], ex["answer"])
for tok, w in zip(d.loss_fn_inputs["target_tokens"].to_torch().tolist(), d.loss_fn_inputs["weights"].to_torch().tolist()):
    print(("\033[32m" if w > 0 else "\033[90m") + tokenizer.decode([tok]), end="\033[0m")
print()

user
How many r's are in strawberry? Provide a numerical answer without units, written inside \boxed{}.<|im_end|>
<|im_start|>assistant
Let's spell the word out and number all the letters: 1) s 2) t 3) r 4) a 5) w 6) b 7) e 8) r 9) r 10) y. We have r's at positions 3, 8, and 9. \boxed{3}<|im_end|>
<|im_start|>user
Mustafa buys a fridge, a TV and a computer with a total budget of $1600. The TV costs $600 and the fridge costs $500 more than the computer. How much does the computer cost? Provide a numerical answer without units, written inside \boxed{}.<|im_end|>
<|im_start|>assistant
<think>

</think>

The fridge and computer cost 1600-600=1000 dollars.
The computer cost (1000-500)/2=250 dollars. \boxed{250}<|im_end|>


In [7]:
# ── LIVE: same loop as 01 ─────────────────────────────────────────────────
training_client = new_training_client()

for step in range(STEPS):
    rows = train.select(range(step * BATCH_SIZE, (step + 1) * BATCH_SIZE))
    batch = [sft_datum(q, a) for q, a in zip(rows["question"], rows["answer"])]
    fwd_bwd = training_client.forward_backward(batch, loss_fn="cross_entropy")
    optim = training_client.optim_step(adam(LR))
    print(f"step {step:2d}  nll/token = {mean_nll(fwd_bwd.result(), batch):.3f}")
    optim.result()

step  0  nll/token = 1.028
step  1  nll/token = 0.736
step  2  nll/token = 0.644
step  3  nll/token = 0.638
step  4  nll/token = 0.622
step  5  nll/token = 0.537


KeyboardInterrupt: 

In [8]:
sampling_client = training_client.save_weights_and_get_sampling_client().result()
eval_gsm8k(sampling_client, label="2. SFT")
save_checkpoint(training_client, "sft")  # RL + distillation can start from here

GSM8K accuracy within 256 tokens [2. SFT]: 78.0% (78/100)   [1 ran out of tokens]
saved 'sft' -> tinker://ffe486f2-cc72-5c61-916c-a8478fecc70f:train:0/weights/sft


'tinker://ffe486f2-cc72-5c61-916c-a8478fecc70f:train:0/weights/sft'

**Takeaway:** SFT = imitation. The model can only become as good as the
demonstrations, and it only ever sees *correct* paths, never its own mistakes.